# 03_ensemble_kfold_from_saved_outputs

This notebook ensembles previously saved model outputs from the modelling notebooks.

It expects each experiment directory to contain:
- `oof_predictions.csv`
- `test_probabilities.csv`

It then:
1. loads OOF and test probabilities
2. aligns them by `image_id`
3. searches ensemble weights
4. tunes the classification threshold on OOF
5. saves:
   - `oof_predictions.csv`
   - `test_probabilities.csv`
   - `submission_best_fold.csv`
   - `submission_fold_ensemble.csv`
   - `metrics_summary.json`
   - `weight_search_results.csv`

In [ ]:

# 1. Config


from pathlib import Path

CONFIG = {
    # --- project / files ---
    "project_root_override": None,
    "root_markers": ["data2", "outputs"],

    # Input experiments to ensemble.
    "input_experiments": [
        {
            "name": "resnet50",
            "relpath": "outputs/experiments/resnet50_etsy_only_v1",
            "oof_proba_col": "oof_ai_generated_proba",
            "test_proba_col": "ai_generated_proba"
        },
        # {
        #     "name": "resnet50_ai5_3",
        #     "relpath": "outputs/experiments/resnet50_etsy_ai5_3_v1",
        #     "oof_proba_col": "oof_ai_generated_proba",
        #     "test_proba_col": "ai_generated_proba",
        # },
        {
            "name": "clip_logreg",
            "relpath": "outputs/experiments/clip_logreg_etsy_only_v1",
            "oof_proba_col": "oof_ai_generated_proba",
            "test_proba_col": "ai_generated_proba",
        },
        # {
        #      "name": "clip_knn",
        #      "relpath": "outputs/experiments/clip_knn_etsy_only_v1",
        #      "oof_proba_col": "oof_ai_generated_proba",
        #      "test_proba_col": "ai_generated_proba",
        # },
        {
             "name": "features_rf",
             "relpath": "outputs/experiments/features_rf_etsy_only_v1",
             "oof_proba_col": "oof_ai_generated_proba",
             "test_proba_col": "ai_generated_proba",
        },
    ],

    "experiment_name": "ensemble_etsy_only_v2",

    # --- labels ---
    "positive_label": "ai_generated",
    "negative_label": "authentic",

    # --- weight search ---
    "weight_step": 0.05,
    "three_model_weight_candidates": [
    # --- ResNet dominant ---
    [0.50, 0.25, 0.25],
    [0.45, 0.275, 0.275],
    [0.40, 0.30, 0.30],
    [0.35, 0.325, 0.325],

    # --- CLIP dominant ---
    [0.25, 0.50, 0.25],
    [0.275, 0.45, 0.275],
    [0.30, 0.40, 0.30],
    [0.325, 0.35, 0.325],

    # --- Features dominant ---
    [0.25, 0.25, 0.50],
    [0.275, 0.275, 0.45],
    [0.30, 0.30, 0.40],
    [0.325, 0.325, 0.35],

    # --- Balanced ---
    [0.333, 0.333, 0.334],
],

    # --- outputs ---
    "save_oof_csv": True,
    "save_test_probs_csv": True,
    "save_submission_csv": True,
}

In [9]:

# 2. Imports and helpers


import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from IPython.display import display


def find_project_root(start: Path, markers: list[str]) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if all((candidate / m).exists() for m in markers):
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. "
        f"Expected markers: {markers}"
    )


def _resolve_path(project_root: Path, p: str | Path) -> Path:
    p = Path(p).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (project_root / p).resolve()


def search_best_threshold(y_true: np.ndarray, y_prob: np.ndarray):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_thr = 0.5
    best_f1 = -1.0

    for thr in thresholds:
        preds = (y_prob >= thr).astype(int)
        score = f1_score(y_true, preds)
        if score > best_f1:
            best_f1 = score
            best_thr = thr

    return float(best_thr), float(best_f1)

In [10]:
# 3. Locate project and load model outputs


cwd = Path.cwd()

if CONFIG["project_root_override"]:
    PROJECT_ROOT = Path(CONFIG["project_root_override"]).expanduser().resolve()
else:
    PROJECT_ROOT = find_project_root(cwd, CONFIG["root_markers"])

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "experiments" / CONFIG["experiment_name"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR  :", OUTPUT_DIR)

oof_frames = []
test_frames = []

for exp in CONFIG["input_experiments"]:
    exp_dir = _resolve_path(PROJECT_ROOT, exp["relpath"])
    oof_path = exp_dir / "reports" / "oof_predictions.csv"
    test_path = exp_dir / "reports" / "test_probabilities.csv"

    if not oof_path.exists():
        raise FileNotFoundError(f"Missing OOF file: {oof_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing test probabilities file: {test_path}")

    oof = pd.read_csv(oof_path)
    test = pd.read_csv(test_path)

    required_oof = {"image_id", "target", exp["oof_proba_col"]}
    required_test = {"image_id", exp["test_proba_col"]}

    miss_oof = required_oof - set(oof.columns)
    miss_test = required_test - set(test.columns)

    if miss_oof:
        raise ValueError(f"{oof_path} missing required columns: {miss_oof}")
    if miss_test:
        raise ValueError(f"{test_path} missing required columns: {miss_test}")

    oof_small = oof[["image_id", "target", exp["oof_proba_col"]]].rename(
        columns={exp["oof_proba_col"]: f"{exp['name']}_oof_proba"}
    )
    test_small = test[["image_id", exp["test_proba_col"]]].rename(
        columns={exp["test_proba_col"]: f"{exp['name']}_test_proba"}
    )

    oof_frames.append(oof_small)
    test_frames.append(test_small)

oof_merged = oof_frames[0].copy()
for frame in oof_frames[1:]:
    oof_merged = oof_merged.merge(frame, on=["image_id", "target"], how="inner")

test_merged = test_frames[0].copy()
for frame in test_frames[1:]:
    test_merged = test_merged.merge(frame, on="image_id", how="inner")

print("OOF merged shape :", oof_merged.shape)
print("Test merged shape:", test_merged.shape)
display(oof_merged.head())
display(test_merged.head())

PROJECT_ROOT: /home/tbeliere/Projects/DCU/AML_PJ
OUTPUT_DIR  : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/ensemble_etsy_only_v1
OOF merged shape : (4800, 5)
Test merged shape: (2058, 4)


,image_id,target,resnet50_oof_proba,clip_logreg_oof_proba,features_rf_oof_proba
0,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,1,1.000000,0.824660,0.6125
1,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,1,0.915362,0.612406,0.4825
2,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,1,0.776160,0.553369,0.5025
3,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,1,0.997034,0.676117,0.4225
4,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,1,0.852163,0.826246,0.8650


,image_id,resnet50_test_proba,clip_logreg_test_proba,features_rf_test_proba
0,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,0.001510,0.167540,0.2795
1,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,0.253792,0.614467,0.7550
2,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,0.034621,0.086512,0.4260
3,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,0.776387,0.605248,0.6935
4,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,0.019811,0.104191,0.2095


In [ ]:
# 4. Build weight candidates

model_names = [exp["name"] for exp in CONFIG["input_experiments"]]
n_models = len(model_names)

if n_models < 2:
    raise ValueError("Need at least 2 models for ensembling.")

weight_candidates = []

if n_models == 2:
    vals = np.arange(0.0, 1.0 + CONFIG["weight_step"], CONFIG["weight_step"])
    for w in vals:
        weight_candidates.append([float(w), float(1.0 - w)])

elif n_models == 3:
    for weights in CONFIG["three_model_weight_candidates"]:
        if len(weights) == 3 and abs(sum(weights) - 1.0) < 1e-8:
            weight_candidates.append([float(w) for w in weights])

else:
    weight_candidates.append([1.0 / n_models] * n_models)
'''
Final candidates after manual tuning:
import pandas as pd
weight_candidates = [
[0.24, 0.27, 0.49],
[0.25, 0.26, 0.49],
[0.26, 0.25, 0.49],
[0.27, 0.24, 0.49],
[0.28, 0.23, 0.49]
]
'''
# print("Models:", model_names)
print("Number of weight candidates:", len(weight_candidates))
display(pd.DataFrame(weight_candidates, columns=model_names).head(20))

Number of weight candidates: 5


,resnet50,clip_logreg,features_rf
0,0.24,0.27,0.49
1,0.25,0.26,0.49
2,0.26,0.25,0.49
3,0.27,0.24,0.49
4,0.28,0.23,0.49


In [42]:
# 5. Weight search on OOF


y_true = oof_merged["target"].astype(int).values
oof_prob_cols = [f"{name}_oof_proba" for name in model_names]
test_prob_cols = [f"{name}_test_proba" for name in model_names]

weight_search_rows = []

for weights in weight_candidates:
    oof_ens = np.zeros(len(oof_merged), dtype=np.float32)
    for w, col in zip(weights, oof_prob_cols):
        oof_ens += w * oof_merged[col].values.astype(np.float32)

    f1_05 = f1_score(y_true, (oof_ens >= 0.5).astype(int))
    best_thr, best_f1 = search_best_threshold(y_true, oof_ens)

    row = {
        "f1_05": float(f1_05),
        "best_threshold": float(best_thr),
        "best_f1": float(best_f1),
    }
    for name, w in zip(model_names, weights):
        row[f"weight__{name}"] = float(w)

    weight_search_rows.append(row)

weight_search_df = pd.DataFrame(weight_search_rows).sort_values(
    ["best_f1", "f1_05"], ascending=False
).reset_index(drop=True)

best_row = weight_search_df.iloc[0].to_dict()
best_weights = [best_row[f"weight__{name}"] for name in model_names]
best_thr = best_row["best_threshold"]

print("Best weights:", dict(zip(model_names, best_weights)))
print("Best threshold:", best_thr)
print("Best OOF F1:", best_row["best_f1"])
display(weight_search_df.head(20))

Best weights: {'resnet50': 0.24, 'clip_logreg': 0.27, 'features_rf': 0.49}
Best threshold: 0.5099999999999998
Best OOF F1: 0.914187887866467


,f1_05,best_threshold,best_f1,weight__resnet50,weight__clip_logreg,weight__features_rf
0,0.912572,0.51,0.914188,0.24,0.27,0.49
1,0.913738,0.51,0.913845,0.25,0.26,0.49
2,0.913034,0.51,0.913183,0.26,0.25,0.49
3,0.910248,0.51,0.912010,0.28,0.23,0.49
4,0.911677,0.51,0.911973,0.27,0.24,0.49


In [14]:
# 6. Final ensemble predictions and evaluation


ensemble_oof = np.zeros(len(oof_merged), dtype=np.float32)
for w, col in zip(best_weights, oof_prob_cols):
    ensemble_oof += w * oof_merged[col].values.astype(np.float32)

ensemble_test = np.zeros(len(test_merged), dtype=np.float32)
for w, col in zip(best_weights, test_prob_cols):
    ensemble_test += w * test_merged[col].values.astype(np.float32)

y_pred_05 = (ensemble_oof >= 0.5).astype(int)
y_pred_best = (ensemble_oof >= best_thr).astype(int)

precision_ai = precision_score(y_true, y_pred_best, pos_label=1)
recall_ai = recall_score(y_true, y_pred_best, pos_label=1)

print("OOF F1 @ 0.5     :", float(f1_score(y_true, y_pred_05)))
print("Best threshold   :", float(best_thr))
print("Best OOF F1      :", float(f1_score(y_true, y_pred_best)))
print("Precision (AI)   :", float(precision_ai))
print("Recall (AI)      :", float(recall_ai))

print("\nClassification report @ best threshold:")
print(classification_report(y_true, y_pred_best, target_names=[CONFIG["negative_label"], CONFIG["positive_label"]]))

print("Confusion matrix @ best threshold:")
print(confusion_matrix(y_true, y_pred_best))

OOF F1 @ 0.5     : 0.893974385891245
Best threshold   : 0.5299999999999998
Best OOF F1      : 0.895618556701031
Precision (AI)   : 0.8906450234942332
Recall (AI)      : 0.9006479481641468

Classification report @ best threshold:
              precision    recall  f1-score   support

   authentic       0.91      0.90      0.90      2485
ai_generated       0.89      0.90      0.90      2315

    accuracy                           0.90      4800
   macro avg       0.90      0.90      0.90      4800
weighted avg       0.90      0.90      0.90      4800

Confusion matrix @ best threshold:
[[2229  256]
 [ 230 2085]]


In [15]:
# 7. Save outputs


id2label = {
    0: CONFIG["negative_label"],
    1: CONFIG["positive_label"],
}

oof_out = pd.DataFrame({
    "image_id": oof_merged["image_id"],
    "target": y_true,
    "oof_ai_generated_proba": ensemble_oof,
})
oof_out["oof_pred_target"] = (oof_out["oof_ai_generated_proba"] >= best_thr).astype(int)
oof_out["oof_pred_label"] = oof_out["oof_pred_target"].map(id2label)

test_out = pd.DataFrame({
    "image_id": test_merged["image_id"],
    "fold_ensemble_ai_generated_proba": ensemble_test,
    "best_fold_ai_generated_proba": ensemble_test,
})

submission_best_fold = pd.DataFrame({
    "image_id": test_merged["image_id"],
    "ground_truth": (ensemble_test >= best_thr).astype(int),
})

submission_fold_ensemble = pd.DataFrame({
    "image_id": test_merged["image_id"],
    "ground_truth": (ensemble_test >= best_thr).astype(int),
})

metrics = {
    "f1_05": float(f1_score(y_true, y_pred_05)),
    "best_threshold": float(best_thr),
    "best_f1": float(f1_score(y_true, y_pred_best)),
    "precision_ai_generated": float(precision_ai),
    "recall_ai_generated": float(recall_ai),
    "n_models": int(n_models),
    "input_models": model_names,
    "best_weights": {name: float(w) for name, w in zip(model_names, best_weights)},
}

if CONFIG["save_oof_csv"]:
    oof_path = OUTPUT_DIR / "oof_predictions.csv"
    oof_out.to_csv(oof_path, index=False)
    print("[SAVE]", oof_path)

if CONFIG["save_test_probs_csv"]:
    test_probs_path = OUTPUT_DIR / "test_probabilities.csv"
    test_out.to_csv(test_probs_path, index=False)
    print("[SAVE]", test_probs_path)

if CONFIG["save_submission_csv"]:
    sub_best_path = OUTPUT_DIR / "submission_best_fold.csv"
    sub_ens_path = OUTPUT_DIR / "submission_fold_ensemble.csv"
    submission_best_fold.to_csv(sub_best_path, index=False)
    submission_fold_ensemble.to_csv(sub_ens_path, index=False)
    print("[SAVE]", sub_best_path)
    print("[SAVE]", sub_ens_path)

metrics_path = OUTPUT_DIR / "metrics_summary.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)
print("[SAVE]", metrics_path)

weight_search_path = OUTPUT_DIR / "weight_search_results.csv"
weight_search_df.to_csv(weight_search_path, index=False)
print("[SAVE]", weight_search_path)

display(oof_out.head())
display(test_out.head())
print(json.dumps(metrics, indent=4))

[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/ensemble_etsy_only_v1/oof_predictions.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/ensemble_etsy_only_v1/test_probabilities.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/ensemble_etsy_only_v1/submission_best_fold.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/ensemble_etsy_only_v1/submission_fold_ensemble.csv
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/ensemble_etsy_only_v1/metrics_summary.json
[SAVE] /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/ensemble_etsy_only_v1/weight_search_results.csv


,image_id,target,oof_ai_generated_proba,oof_pred_target,oof_pred_label
0,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,1,0.814263,1,ai_generated
1,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,1,0.672542,1,ai_generated
2,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,1,0.612331,1,ai_generated
3,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,1,0.701535,1,ai_generated
4,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,1,0.847847,1,ai_generated


,image_id,fold_ensemble_ai_generated_proba,best_fold_ai_generated_proba
0,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,0.148037,0.148037
1,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,0.538213,0.538213
2,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,0.180900,0.180900
3,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,0.692558,0.692558
4,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,0.110254,0.110254


{
    "f1_05": 0.893974385891245,
    "best_threshold": 0.5299999999999998,
    "best_f1": 0.895618556701031,
    "precision_ai_generated": 0.8906450234942332,
    "recall_ai_generated": 0.9006479481641468,
    "n_models": 3,
    "input_models": [
        "resnet50",
        "clip_logreg",
        "features_rf"
    ],
    "best_weights": {
        "resnet50": 0.34,
        "clip_logreg": 0.33,
        "features_rf": 0.33
    }
}
